# Failure Mode Exploration

This notebook is intentionally about mistakes.

In an educational agent project, failure cases are not side material. They are one of the best ways to understand the limits of the tools, the prompt, and the ReAct policy.

## Step 1: Run The Agent Across The Materialized Dataset

This cell executes the experiment runner against the saved dataset rather than generating a fresh sample.

Why this matters:

- it makes the bad cases reproducible
- it lets you revisit the exact same campaigns after changing prompts or heuristics
- it keeps the debugging loop grounded in stable examples


In [ ]:
from iroas_agent.runner import run_experiment

In [ ]:
output = run_experiment(dataset_path='data/synthetic_campaigns.csv', sample_size=20)
output['summary_metrics']

## Step 2: Pull Out The Worst Errors

This cell sorts the runs by absolute iROAS error.

Why this matters:

- the largest mistakes often reveal the most important design issues
- they show whether the agent is over-trusting a weak estimator or under-using good causal evidence
- they also show whether the stopping policy is too eager


In [ ]:
bad_cases = sorted(
    output['results'],
    key=lambda row: abs(row['prediction']['estimated_iROAS'] - row['metadata']['true_iROAS']),
    reverse=True,
)[:5]
bad_cases[0]['prediction']

## Step 3: Inspect One Failure In Detail

This cell prints a compact summary of the worst case.

What to look for:

- which estimator the agent finally trusted
- how large the gap is between predicted and true iROAS
- whether the campaign had stronger evidence available that the agent failed to use well


In [ ]:
{
    'campaign_id': bad_cases[0]['campaign_id'],
    'predicted_iROAS': bad_cases[0]['prediction']['estimated_iROAS'],
    'true_iROAS': bad_cases[0]['metadata']['true_iROAS'],
    'final_estimator_used': bad_cases[0]['prediction']['final_estimator_used'],
    'data_availability_type': bad_cases[0]['metadata']['data_availability_type'],
}

## Step 4: Read The Failure Trajectory

This final cell prints the reasoning trace for the worst case.

Why this matters:

- a bad final number becomes much more actionable when you can see the exact Thought -> Action -> Observation chain that produced it
- this is often where prompt improvements become obvious
- it also tells you whether the underlying estimator or the agent policy is the more urgent problem


In [ ]:
for step in bad_cases[0]['trajectory']:
    print('Thought:', step['thought'])
    print('Action:', step['action'])
    print('Observation:', step['observation'])
    print('-' * 80)

## What To Take Away

Failure analysis is where the educational value compounds.

When you can connect a large error to a concrete reasoning path, you get a much clearer sense of what to improve next.